# 311 API Data

## Author: Yi Wang
## Group: Yi Wang, An Nisa Astuti

### Notebook Description: 
This notebook pulls Chicago 311 service request records from the City of Chicago’s Socrata API (v6vf-nfxy dataset) in monthly chunks from 2021–2025. It saves the raw CSVs, then concatenates and cleans them by dates and community area. We compute operational performance metrics including request volume, closure/share open, time-to-close (TTC) stats, open-age backlog stats, and % closed within 24h/72h/7d/30d. We then grouped these metrics by community_area × year × ward, and writes the results to data/311_metrics.csv.

### AI Statement
I used ChatGPT to learn a new API documentation called Socrata and used AI for understanding the psuedo code from https://dev.socrata.com/foundry/data.cityofchicago.org/v6vf-nfxy, including setting query parameters, handling pagination, and troubleshooting HTTP/API errors. 

In [9]:
import requests, pandas as pd, sqlite3
from bs4 import BeautifulSoup as bs
from sodapy import Socrata

In [10]:
url = "https://data.cityofchicago.org/resource/v6vf-nfxy.json"
response = requests.get(url)
print("Our response code is:", response.status_code) 
# check response code to make sure it works 

Our response code is: 200


### Testing with the token

In [11]:
# I am citing this from Chicago data portal 311 API documentation and socrata API code snippets (Chicago Open Data portal uses Socrata)
# On the code snippets, it has instructions and code on how to access the API data with token
# using the original template, I am building on the given template
# "https://dev.socrata.com/foundry/data.cityofchicago.org/v6vf-nfxy"
# "https://dev.socrata.com/docs/queries/page"


token = "iAZp9mxjcw3dt8SUNugTPYJbS" 
client = Socrata("data.cityofchicago.org", token, timeout = 60)

DATASET_ID = "v6vf-nfxy"

SELECT_COLS = ",".join(["sr_number",
    "sr_type",
    "sr_short_code",
    "status",
    "origin",
    "created_department",
    "owner_department",
    "created_date",
    "last_modified_date",
    "closed_date",
    "latitude",
    "longitude",
    "zip_code",
    "community_area",
    "ward",
    "duplicate",
    "legacy_record"
])

BASE_WHERE = """
created_date >= '{start}' AND created_date < '{end}'
AND duplicate = false AND legacy_record = false
"""

def fetch_311(start, end, limit=50000):
    """Fetch ALL rows for [start, end) using Socrata paging."""
    offset = 0
    out = []

    where = BASE_WHERE.format(start=start, end=end)

    while True:
        rows = client.get(
            DATASET_ID,
            select=SELECT_COLS,
            where=where,
            order="created_date ASC, sr_number ASC",
            limit=limit,
            offset=offset
        )
        if not rows:
            break

        out.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit

    if out:
        return pd.concat(out, ignore_index=True)
    return pd.DataFrame()


### Pulling data in chunks

In [12]:
import time

def fetch_all(where, limit=20000, max_retries=5):
    offset = 0
    frames = []

    while True:
        attempt = 0
        while True:
            try:
                rows = client.get(
                    DATASET_ID,
                    select=SELECT_COLS,
                    where=where,
                    order="created_date ASC, sr_number ASC",
                    limit=limit,
                    offset=offset
                )
                break  
            except Exception as e:
                attempt += 1
                if attempt > max_retries:
                    raise  

                wait = 2 * attempt 
                print(f"Timeout, trying...")
                time.sleep(wait)

        if not rows:
            break

        frames.append(pd.DataFrame.from_records(rows))

        if len(rows) < limit:
            break

        offset += limit
        time.sleep(0.2) 

    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()

In [14]:
where_jan2023 = BASE_WHERE.format(
    start="2023-01-01T00:00:00.000",
    end="2023-02-01T00:00:00.000"
)

df_jan = fetch_all(where_jan2023, limit=50000)
print("Jan 2023 rows:", len(df_jan))
df_jan.head()

KeyboardInterrupt: 

- Missing rate of zipcode is too high, we have decided to switch from using zipcode to community area and ward number. 

In [15]:
df_test = pd.read_csv("data/raw_data/311_2023_01.csv")
print(df_test["zip_code"].isna().mean())
print(df_test["zip_code"].value_counts(dropna=False).head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw_data/311_2023_01.csv'

In [ ]:
def next_month(y, m):
    if m == 12:
        return (y + 1, 1)
    return (y, m + 1)

for y in [2021, 2022, 2023, 2024, 2025]:
    for m in range(1, 13):
        y2, m2 = next_month(y, m)

        start = f"{y}-{m:02d}-01T00:00:00.000"
        end   = f"{y2}-{m2:02d}-01T00:00:00.000"
        tag   = f"{y}_{m:02d}"

        where_m = BASE_WHERE.format(start=start, end=end)
        df_m = fetch_all(where_m, limit=20000)

        print(tag, "rows:", len(df_m))
        df_m.to_csv(f"data/raw_data/311_{tag}.csv", index=False)

### Data Cleaning

In [ ]:
import glob

files = sorted(glob.glob("data/raw_data/311_*.csv"))
dfs = [pd.read_csv(f) for f in files]
df_311 = pd.concat(dfs, ignore_index=True)

print("total:", len(df_311))
df_311.head()

In [ ]:
def add_basic_metrics_cols(df):
    df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
    df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")

    df["year"] = df["created_date"].dt.year

    df["is_closed"] = df["closed_date"].notna()

    df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
    df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA  

    return df


In [ ]:
df_311.to_csv("data/311_2021_to_2025_clean.csv", index=False)

In [ ]:
df = pd.read_csv("data/311_2021_to_2025_clean.csv")

df["community_area"] = pd.to_numeric(df["community_area"], errors="coerce")
df = df.dropna(subset=["community_area"]).copy()
df["community_area"] = df["community_area"].astype(int)

df["created_date"] = pd.to_datetime(df["created_date"], errors="coerce")
df["closed_date"]  = pd.to_datetime(df["closed_date"], errors="coerce")
df = df.dropna(subset=["created_date"]).copy()
df["year"] = df["created_date"].dt.year

df["is_closed"] = df["closed_date"].notna()

# TTC for closed only
df["ttc_hours"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 3600
df.loc[~df["is_closed"], "ttc_hours"] = pd.NA
df.loc[df["ttc_hours"] < 0, "ttc_hours"] = pd.NA

ASOF = df["created_date"].max()
df["age_open_hours"] = (ASOF - df["created_date"]).dt.total_seconds() / 3600
df.loc[df["is_closed"], "age_open_hours"] = pd.NA
df.loc[df["age_open_hours"] < 0, "age_open_hours"] = pd.NA

# Precompute closure-within indicators (open requests count as False)
df["closed_within_24h"] = df["is_closed"] & (df["ttc_hours"] <= 24)
df["closed_within_72h"] = df["is_closed"] & (df["ttc_hours"] <= 72)
df["closed_within_7d"]  = df["is_closed"] & (df["ttc_hours"] <= 24*7)
df["closed_within_30d"] = df["is_closed"] & (df["ttc_hours"] <= 24*30)

final_metrics = (
    df.groupby(["community_area", "year", "ward"], dropna=False)
      .agg(
          n_requests=("sr_number", "size"),
          n_closed=("is_closed", "sum"),
          share_closed=("is_closed", "mean"),

          n_open=("is_closed", lambda s: (~s).sum()),
          share_open=("is_closed", lambda s: (~s).mean()),

          median_ttc_hours=("ttc_hours", "median"),
          p75_ttc_hours=("ttc_hours", lambda s: s.quantile(0.75)),
          mean_ttc_hours=("ttc_hours", "mean"),

          median_age_open_hours=("age_open_hours", "median"),
          p75_age_open_hours=("age_open_hours", lambda s: s.quantile(0.75)),

          pct_closed_24h=("closed_within_24h", "mean"),
          pct_closed_72h=("closed_within_72h", "mean"),
          pct_closed_7d=("closed_within_7d", "mean"),
          pct_closed_30d=("closed_within_30d", "mean"),
      )
      .reset_index()
)

final_metrics["small_n_requests"] = final_metrics["n_requests"] < 30
final_metrics["small_n_closed"]   = final_metrics["n_closed"] < 30

# Round readable columns
to_round = [
    "share_closed", "share_open",
    "median_ttc_hours", "p75_ttc_hours", "mean_ttc_hours",
    "median_age_open_hours", "p75_age_open_hours",
    "pct_closed_24h", "pct_closed_72h", "pct_closed_7d", "pct_closed_30d",
]
for c in to_round:
    if c in final_metrics.columns:
        final_metrics[c] = final_metrics[c].round(4)

final_metrics.to_csv("data/311_metrics.csv", index=False)
print("Saved:", final_metrics.shape)
final_metrics.head()


In [ ]:
print(len(df)) # check length
print(df["created_date"].min(), "to", df["created_date"].max()) # check time period 
print("missing community_area:", df["community_area"].isna().mean()) # check missing rate
print("missing closed_date:", df["closed_date"].isna().mean()) # check missing rate for closed_date for request

final_metrics.sort_values("n_requests", ascending=False).head(10)


### Reading the new compiled and cleaned file

In [8]:
df2 = pd.read_csv("data/311_metrics.csv")  

FileNotFoundError: [Errno 2] No such file or directory: 'data/311_metrics.csv'

In [ ]:
df2.columns